# 📝 Module 02: Prompt Templates

---

## What Are Prompt Templates?

**Prompt Templates** are reusable, parameterized prompts. Instead of hardcoding prompts, you define a **template with variables** and fill them in at runtime.

### Without Templates (Bad Practice)
```python
# Hardcoded — hard to reuse, messy to maintain
prompt = f"Translate '{user_text}' from {source_lang} to {target_lang}."
```

### With Templates (Best Practice)
```python
# Clean, reusable, testable
template = PromptTemplate.from_template(
    "Translate '{text}' from {source} to {target}."
)
prompt = template.invoke({"text": "Hello", "source": "English", "target": "French"})
```

---

## Types of Prompt Templates

| Template Type | Use Case | Output |
|---------------|----------|--------|
| `PromptTemplate` | Single string prompts | String |
| `ChatPromptTemplate` | Chat model prompts | List of Messages |
| `FewShotPromptTemplate` | Few-shot learning | Prompt with examples |
| `FewShotChatMessagePromptTemplate` | Few-shot for chat | Chat messages with examples |

---

In [6]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)
print("Setup complete ✅")

Setup complete ✅


## 1️⃣ PromptTemplate — Basic String Templates

In [7]:
from langchain_core.prompts import PromptTemplate

# ============================================================
# Method 1: from_template() — Simple and most common
# ============================================================
template = PromptTemplate.from_template(
    "Write a {style} explanation of {topic} for a {audience}."
)

# Check the input variables automatically detected
print("Input variables:", template.input_variables)

# Format the template
formatted = template.invoke({
    "style": "simple and fun",
    "topic": "machine learning",
    "audience": "5-year-old"
})

print("\nFormatted prompt:")
print(formatted.text)

Input variables: ['audience', 'style', 'topic']

Formatted prompt:
Write a simple and fun explanation of machine learning for a 5-year-old.


In [8]:
# ============================================================
# Method 2: Constructor with explicit variables
# ============================================================
template2 = PromptTemplate(
    input_variables=["product", "tone"],
    template="Write a {tone} product description for {product}. Keep it under 50 words."
)

# Use in a chain!
chain = template2 | llm
response = chain.invoke({"product": "AI-powered coffee maker", "tone": "exciting and energetic"})
print(response.content)

"Brew genius with our AI-powered coffee maker, expertly crafting the perfect cup every time with precision temperature control and flavor profiling!"


In [9]:
# ============================================================
# Partial Templates — Pre-fill some variables
# ============================================================
template = PromptTemplate.from_template(
    "You are a {role}. Answer this {language} question: {question}"
)

# Pre-fill 'role' for a specific use case
python_expert_template = template.partial(role="Python expert", language="Python")

# Now only 'question' needs to be provided
chain = python_expert_template | llm
response = chain.invoke({"question": "What is a generator?"})
print(response.content)

**Generators in Python**

A generator is a special type of function in Python that can be used to generate a sequence of results instead of computing them all at once and returning them in a list, for example.

**Key Characteristics:**

*   Generators are defined using functions and the `yield` keyword.
*   When a generator is called, it returns an iterator object.
*   Generators compute their values on-the-fly, which means they only create objects in memory as needed.
*   Generators can be used in a loop to retrieve values one at a time.

**Example:**
```python
def infinite_sequence():
    num = 0
    while True:
        yield num
        num += 1

# Create a generator
gen = infinite_sequence()

# Print the first 10 values
for _ in range(10):
    print(next(gen))
```
In this example, `infinite_sequence` is a generator that produces an infinite sequence of numbers. The `yield` keyword is used to produce a value, and the function remembers its state between calls.

**Benefits:**

*   **

## 2️⃣ ChatPromptTemplate — For Chat Models

In [10]:
from langchain_core.prompts import ChatPromptTemplate

# ============================================================
# Basic ChatPromptTemplate
# ============================================================
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a {role} with expertise in {domain}. Be {style}."),
    ("human", "{question}")
])

# Inspect the template
print("Input variables:", chat_template.input_variables)

# Format and send
chain = chat_template | llm
response = chain.invoke({
    "role": "senior software engineer",
    "domain": "distributed systems",
    "style": "concise and technical",
    "question": "What is the CAP theorem?"
})

print(response.content)

Input variables: ['domain', 'question', 'role', 'style']
**CAP Theorem**: In a distributed data storage system, it is impossible to simultaneously guarantee all three of the following:

1. **Consistency**: Every read operation will see the most recent write or an error.
2. **Availability**: Every request receives a response, without guarantee that it contains the most recent version of the information.
3. **Partition Tolerance**: The system continues to function and make progress even when network partitions (i.e., splits or failures) occur.

At most two of these properties can be guaranteed at the same time. This fundamental trade-off is known as the CAP theorem.


In [11]:
# ============================================================
# Multi-turn Conversation Template
# ============================================================
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# MessagesPlaceholder lets you inject a full list of messages
multi_turn_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful coding assistant."),
    MessagesPlaceholder(variable_name="history"),  # Past messages go here
    ("human", "{current_question}")
])

# Simulate a conversation history
history = [
    HumanMessage(content="I'm learning Python."),
    AIMessage(content="Great! Python is a fantastic language to learn."),
]

chain = multi_turn_template | llm
response = chain.invoke({
    "history": history,
    "current_question": "What should I learn after basics?"
})

print(response.content)

After learning the basics of Python, here are some topics you may want to explore:

1. **Data Structures**: Learn about lists, tuples, dictionaries, sets, and other data structures in Python. Understanding how to work with these data structures is crucial for any Python programmer.
2. **File Input/Output**: Learn how to read and write files in Python, including text files, CSV files, and JSON files.
3. **Functions**: Learn how to define and use functions in Python, including function arguments, return types, and scope.
4. **Object-Oriented Programming (OOP)**: Learn about classes, objects, inheritance, polymorphism, and encapsulation in Python.
5. **Error Handling**: Learn how to handle errors and exceptions in Python using try-except blocks and error handling mechanisms.
6. **Modules and Packages**: Learn how to import and use external modules and packages in Python, including popular libraries like NumPy, Pandas, and Requests.
7. **Data Analysis and Visualization**: Learn how to work

In [12]:
# ============================================================
# Using Message Objects directly
# ============================================================
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage

# Mix of message objects and tuples
template = ChatPromptTemplate.from_messages([
    SystemMessage(content="Always respond in bullet points."),
    ("human", "List the top {n} benefits of {topic}")
])

chain = template | llm
response = chain.invoke({"n": 5, "topic": "meditation"})
print(response.content)

* Reduces stress and anxiety: Meditation has been shown to decrease the production of stress hormones like cortisol, leading to a sense of calm and relaxation.
* Improves sleep: Regular meditation practice can help improve sleep quality, duration, and depth, leading to better overall health and well-being.
* Increases focus and concentration: Meditation can improve attention and focus by training the mind to stay present and aware, leading to greater productivity and efficiency.
* Boosts mood and emotional well-being: Meditation has been linked to increased production of neurotransmitters like serotonin and dopamine, which can help alleviate symptoms of depression and anxiety.
* Enhances self-awareness and emotional regulation: Meditation can help individuals develop a greater understanding of themselves, their thoughts, and their emotions, leading to better emotional regulation and decision-making.


## 3️⃣ Few-Shot Prompt Templates — Learning from Examples

In [13]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

# ============================================================
# Few-shot examples: Sentiment analysis
# ============================================================
examples = [
    {"review": "This product is amazing! Love it.", "sentiment": "POSITIVE"},
    {"review": "Terrible quality. Broke after one use.", "sentiment": "NEGATIVE"},
    {"review": "It's okay. Nothing special.", "sentiment": "NEUTRAL"},
    {"review": "Best purchase I've ever made!", "sentiment": "POSITIVE"},
]

# Template for each example
example_template = PromptTemplate(
    input_variables=["review", "sentiment"],
    template="Review: {review}\nSentiment: {sentiment}"
)

# Assemble the few-shot template
few_shot_template = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_template,
    prefix="Classify the sentiment of product reviews. Use POSITIVE, NEGATIVE, or NEUTRAL.",
    suffix="Review: {new_review}\nSentiment:",
    input_variables=["new_review"],
    example_separator="\n\n"
)

# See the full assembled prompt
print("Full prompt preview:")
print("-" * 60)
print(few_shot_template.format(new_review="Great but a bit expensive."))

Full prompt preview:
------------------------------------------------------------
Classify the sentiment of product reviews. Use POSITIVE, NEGATIVE, or NEUTRAL.

Review: This product is amazing! Love it.
Sentiment: POSITIVE

Review: Terrible quality. Broke after one use.
Sentiment: NEGATIVE

Review: It's okay. Nothing special.
Sentiment: NEUTRAL

Review: Best purchase I've ever made!
Sentiment: POSITIVE

Review: Great but a bit expensive.
Sentiment:


In [14]:
# Run it!
chain = few_shot_template | llm

test_reviews = [
    "Absolutely love this! Five stars.",
    "Disappointed. Doesn't work as advertised.",
    "It works fine for the price."
]

for review in test_reviews:
    response = chain.invoke({"new_review": review})
    print(f"Review: '{review}'")
    print(f"Sentiment: {response.content.strip()}\n")

Review: 'Absolutely love this! Five stars.'
Sentiment: POSITIVE

Review: 'Disappointed. Doesn't work as advertised.'
Sentiment: NEGATIVE

Review: 'It works fine for the price.'
Sentiment: NEUTRAL



## 4️⃣ Few-Shot Chat Prompt Templates

In [21]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate, ChatPromptTemplate

# Examples as conversations
examples = [
    {"input": "2 + 2", "output": "4"},
    {"input": "10 * 5", "output": "50"},
    {"input": "100 / 4", "output": "25"},
]

# Template for a single example
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "Calculate: {input}"),
    ("ai", "{output}")
])

# Few-shot chat template
few_shot = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

# Final template wrapping the few-shot block
final_template = ChatPromptTemplate.from_messages([
    ("system", "You are a math calculator. Only return the numeric answer."),
    few_shot,
    ("human", "Calculate: {question}")
])

chain = final_template | llm
response = chain.invoke({"question": "15 * 10 - 19"})
print(f"Answer: {response.content}")

Answer: 131


## 5️⃣ Advanced Template Features

In [22]:
# ============================================================
# Dynamic Partial Variables — Use functions as defaults
# ============================================================
from datetime import datetime

def get_current_date():
    return datetime.now().strftime("%B %d, %Y")

template = PromptTemplate(
    input_variables=["query"],
    partial_variables={"date": get_current_date},  # Function called at runtime!
    template="Today is {date}. Answer this query: {query}"
)

chain = template | llm
response = chain.invoke({"query": "What day of the week is it?"})
print(response.content)

May 26, 2026, is a Tuesday.


In [23]:
# ============================================================
# Template Composition — Reuse templates in other templates
# ============================================================

# Base template pieces
persona_template = "You are a {persona}.\n"
style_template = "Always respond in {style} style.\n"
task_template = "Task: {task}"

# Compose them together
full_template = ChatPromptTemplate.from_template(
    persona_template + style_template + task_template
)

chain = full_template | llm
response = chain.invoke({
    "persona": "Shakespearean bard",
    "style": "poetic and dramatic",
    "task": "Explain why Python is the best programming language"
})
print(response.content)

O, fairest of coders, gather 'round and heed my tale,
Of Python, a language most divine, beyond fail.
'Tis a tongue of elegance, with syntax so fine,
That doth entwine the heart of programmer and machine in love's sweet bind.

Its indentation, a gentle guide, doth lead the way,
Through labyrinths of code, where errors oft do stray.
The whitespace, a beacon bright, doth shine like Phoebus' ray,
 Illuminating paths, where darkness once did hold its sway.

The libraries, a treasure trove, of wonders to behold,
Doth grant the coder, powers to shape and mold.
From data's mighty streams, to webs that doth unfold,
Python's magic, doth weave a tapestry, of wonders to behold.

The community, a fellowship, of noble hearts and minds,
Doth gather 'neath the banner, of Python's noble finds.
The documentation, a tome, of wisdom and of might,
Doth stand as sentinel, guarding the gates of night.

And when the coder, doth stumble, and errors doth abound,
The Pythonic spirit, doth whisper, "Fear not, de

## 6️⃣ Prompt Engineering Best Practices

### 🎯 The CRAFT Framework

| Letter | Stands For | Example |
|--------|-----------|----------|
| **C** | Context | "You are a senior Python developer..." |
| **R** | Role | "...with 10 years of experience" |
| **A** | Action | "Explain the following code" |
| **F** | Format | "Use bullet points with code examples" |
| **T** | Tone | "Be concise and technical" |

### 📋 Template Best Practices

```python
# ✅ GOOD: Clear structure with role, task, format
good_template = ChatPromptTemplate.from_messages([
    ("system", """
    You are an expert {domain} specialist.
    
    When answering:
    - Be specific and accurate
    - Use examples when helpful
    - If you're unsure, say so
    - Format code in markdown blocks
    """),
    ("human", "{question}")
])

# ❌ BAD: Vague and unstructured
bad_template = PromptTemplate.from_template("Tell me about {topic}")
```

In [ ]:
# ============================================================
# Chain-of-Thought Prompting
# ============================================================
cot_template = ChatPromptTemplate.from_messages([
    ("system", """
    You are a logical reasoning assistant.
    For every question:
    1. Think step by step
    2. Show your reasoning
    3. Conclude with a clear answer
    """),
    ("human", "{problem}")
])

chain = cot_template | llm
response = chain.invoke({
    "problem": "If a train travels 120 km/h and needs to cover 300 km, but stops for 30 minutes, how long does the total trip take?"
})
print(response.content)

## ✅ Module 02 Summary

You've learned:
- ✅ `PromptTemplate` for string-based prompts
- ✅ `ChatPromptTemplate` for chat model prompts
- ✅ `MessagesPlaceholder` for dynamic conversation history
- ✅ Few-shot prompting with examples
- ✅ Partial templates and dynamic variables
- ✅ Chain-of-thought prompting patterns
- ✅ The CRAFT framework for prompt engineering

### 🚀 Next: [Module 03 — Output Parsers](03_Output_Parsers.ipynb)